<a href="https://colab.research.google.com/github/gyanadhaanainar/signlanguage_recognition/blob/main/signlanguage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_DIR = (
    "/content/drive/MyDrive/SignLanguageDetection/dataset"
)

In [ ]:
!cp -r /content/drive/MyDrive/SignLanguageDetection/asl_alphabet_train /content/data

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models


In [ ]:
DATA_DIR = "/content/drive/MyDrive/SignLanguageDetection/asl_alphabet_train/asl_alphabet_train"

IMG_SIZE = (64, 64)
BATCH_SIZE = 32
SEED = 123

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)


In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
import os
class_names = sorted([
    folder for folder in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, folder))
])

num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)

In [ ]:
num_classes = len(class_names)

model = models.Sequential([
    layers.Input(shape=(64, 64, 3)),

    # Normalize pixels from 0-255 to 0-1
    layers.Rescaling(1.0 / 255),

    # First convolution block
    layers.Conv2D(
        32, (3, 3), activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Second convolution block
    layers.Conv2D(
        64, (3, 3), activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Third convolution block
    layers.Conv2D(
        128, (3, 3), activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Classification layers
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop]
)


In [ ]:
model.save("sign_language_cnn.keras")
print("Model saved successfully!")

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

svm_model = SVC(kernel="rbf", C=5)
svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)

print("SVM Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(
    y_test, svm_pred,
    labels=np.arange(len(class_names_ml)),
    target_names=class_names_ml,
    zero_division=0
))

In [ ]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.utils import load_img, img_to_array

model = tf.keras.models.load_model(
    "sign_language_cnn.keras"
)

# Must match the class folder names used during training
class_names = [
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space'
]

image_path = "my_sign.jpg"

img = load_img(
    image_path,
    target_size=(64, 64)
)

img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

predictions = model.predict(img_array, verbose=0)[0]

predicted_index = np.argmax(predictions)
predicted_sign = class_names[predicted_index]
confidence = predictions[predicted_index] * 100

print("Predicted sign:", predicted_sign)
print(f"Confidence: {confidence:.2f}%")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    adjusted_rand_score, normalized_mutual_info_score,
    silhouette_score
)

In [ ]:
IMG_SIZE_ML = (32, 32)
SAMPLES_PER_CLASS = 100

X = []
y = []

for class_name in class_names:
    class_path = os.path.join(DATA_DIR, class_name)
    images = os.listdir(class_path)[:SAMPLES_PER_CLASS]

    for image_file in images:
        image_path = os.path.join(class_path, image_file)

        image = tf.keras.utils.load_img(
            image_path,
            color_mode='grayscale',
            target_size=IMG_SIZE_ML
        )

        image = tf.keras.utils.img_to_array(image)
        X.append(image.flatten())
        y.append(class_names.index(class_name))

X = np.array(X)
y = np.array(y)

print("Dataset Shape:", X.shape)
print("Labels Shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Shape:", X_train_scaled.shape)
print("Testing Shape:", X_test_scaled.shape)

In [ ]:
selector = SelectKBest(score_func=f_classif, k=100)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

print("Original Features:", X_train_scaled.shape[1])
print("Selected Features:", X_train_selected.shape[1])

In [ ]:
pca = PCA(n_components=50)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original Features:", X_train_scaled.shape[1])
print("PCA Components:", X_train_pca.shape[1])
print("Explained Variance:", sum(pca.explained_variance_ratio_))

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    np.cumsum(pca.explained_variance_ratio_),
    marker='o'
)

plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Explained Variance')
plt.grid()
plt.show()

In [ ]:
  lda = LinearDiscriminantAnalysis(
    n_components=min(len(np.unique(y_train)) - 1, X_train_scaled.shape[1])
)

X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda = lda.transform(X_test_scaled)

print("LDA Training Shape:", X_train_lda.shape)
print("LDA Testing Shape:", X_test_lda.shape)

In [ ]:
dt_gini = DecisionTreeClassifier(
    criterion='gini',
    random_state=42
)

dt_gini.fit(X_train_scaled, y_train)

y_pred_gini = dt_gini.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred_gini))
print(classification_report(y_test, y_pred_gini, zero_division=0))

In [ ]:
dt_entropy = DecisionTreeClassifier(
    criterion='entropy',
    random_state=42
)

dt_entropy.fit(X_train_scaled, y_train)

y_pred_entropy = dt_entropy.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred_entropy))
print(classification_report(y_test, y_pred_entropy, zero_division=0))

In [ ]:
from math import log2

def calculate_entropy(labels):
    counts = np.bincount(labels)
    probabilities = counts[counts > 0] / len(labels)

    return -sum(p * log2(p) for p in probabilities)

root_entropy = calculate_entropy(y_train)

tree = dt_entropy.tree_

feature = tree.feature[0]
threshold = tree.threshold[0]

left = X_train_scaled[:, feature] <= threshold
right = X_train_scaled[:, feature] > threshold

left_entropy = calculate_entropy(y_train[left])
right_entropy = calculate_entropy(y_train[right])

weighted_entropy = (
    (sum(left) / len(y_train)) * left_entropy +
    (sum(right) / len(y_train)) * right_entropy
)

information_gain = root_entropy - weighted_entropy

print("Root Entropy:", root_entropy)
print("Information Gain:", information_gain)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    criterion='entropy',
    random_state=42
)

rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, zero_division=0))

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=30,
    learning_rate=0.1,
    max_depth=2,
    random_state=42
)

gb.fit(X_train_pca, y_train)

y_pred_gb = gb.predict(X_test_pca)

print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb, zero_division=0))

In [ ]:
svm = SVC(kernel='rbf', C=5)

svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm, zero_division=0))

In [ ]:
svm_pca = SVC(kernel='rbf', C=5)

svm_pca.fit(X_train_pca, y_train)

y_pred_svm_pca = svm_pca.predict(X_test_pca)

print("SVM with PCA Accuracy:", accuracy_score(y_test, y_pred_svm_pca))

In [ ]:
models = [
    'Decision Tree Gini',
    'Decision Tree Entropy',
    'Random Forest',
    'Gradient Boosting',
    'SVM',
    'SVM PCA'
]

predictions = [
    y_pred_gini,
    y_pred_entropy,
    y_pred_rf,
    y_pred_gb,
    y_pred_svm,
    y_pred_svm_pca
]

results = []

for name, pred in zip(models, predictions):

    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, average='weighted', zero_division=0),
        recall_score(y_test, pred, average='weighted', zero_division=0),
        f1_score(y_test, pred, average='weighted', zero_division=0)
    ])

results_df = pd.DataFrame(
    results,
    columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score']
)

results_df

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(results_df['Model'], results_df['Accuracy'])

plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Comparison of Classification Models')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
wcss = []

for k in range(2, 30, 2):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_train_pca)

    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))

plt.plot(range(2, 30, 2), wcss, marker='o')

plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('Elbow Method')

plt.grid()
plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=len(class_names),
    random_state=42,
    n_init=10
)

kmeans.fit(X_train_pca)

kmeans_labels = kmeans.labels_

print("Adjusted Rand Index:",
      adjusted_rand_score(y_train, kmeans_labels))

print("Normalized Mutual Information:",
      normalized_mutual_info_score(y_train, kmeans_labels))

print("Silhouette Score:",
      silhouette_score(X_train_pca, kmeans_labels))

In [ ]:
sample_size = min(1500, len(X_train_pca))

indices = np.random.RandomState(42).choice(
    len(X_train_pca),
    sample_size,
    replace=False
)

X_dbscan = X_train_pca[indices]
y_dbscan = y_train[indices]

dbscan_scaler = StandardScaler()

X_dbscan_scaled = dbscan_scaler.fit_transform(X_dbscan)

dbscan = DBSCAN(eps=1.5, min_samples=5)

dbscan_labels = dbscan.fit_predict(X_dbscan_scaled)

clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)

print("Number of Clusters:", clusters)
print("Noise Points:", list(dbscan_labels).count(-1))

if clusters > 1:
    non_noise = dbscan_labels != -1

    print("Adjusted Rand Index:",
          adjusted_rand_score(y_dbscan[non_noise], dbscan_labels[non_noise]))

    print("Normalized Mutual Information:",
          normalized_mutual_info_score(y_dbscan[non_noise], dbscan_labels[non_noise]))
else:
    print("Not enough clusters found. Try changing eps.")

In [ ]:
results_df.to_csv(
    '/content/drive/MyDrive/ASL_model_comparison.csv',
    index=False
)

print("Results saved successfully!")